# Core-preserving simplification QLoRA

Train a question-only Qwen3-4B simplifier from an existing Phase 1 donor JSON and its matching run log. `SUCCESS` learns the accepted proxy; `REJECTED_BY_FILTER` and `SKIPPED_FAILSAFE` learn to copy the original. Solver evaluation compares direct solving with base and adapted simplification on held-out construction questions.

In [ ]:
# Run from a Kaggle GPU notebook with this repository available at REPO_DIR.
from pathlib import Path
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO_DIR = Path("/kaggle/working/analogical_math_rag")
if (Path.cwd() / "src").is_dir():
    REPO_DIR = Path.cwd()
if not (REPO_DIR / "src").is_dir():
    raise FileNotFoundError(f"Repository not found: {REPO_DIR}")
os.chdir(REPO_DIR)
%pip install -q -r requirements-merging-finetuning.txt

## Configuration

Set the Hugging Face **dataset** repository ID and the two relative filenames from the same Phase 1 run. A private repository needs a read-enabled `HF_TOKEN` in Kaggle Secrets. The evaluator needs `AVALAI_API_KEY` when evaluation is enabled.

In [ ]:
import json, random
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer
from src.merging_finetuning import (
    QLoRAConfig, CompletionOnlyCollator, LocalFusionGenerator,
    load_adapter_for_inference, load_qlora_model, smoke_test_training_step,
    tokenize_splits, train_qlora, upload_adapter_to_hub,
)
from src.simplification_finetuning import (
    prepare_simplification_data, evaluate_question, summarize_evaluation,
)
from src.utils import save_json_atomic
from config import CONFIG, setup_kaggle_mode

assert torch.cuda.is_available(), "Select a Kaggle GPU accelerator."
print(torch.cuda.get_device_name(0))

HF_DATASET_REPO_ID = "owner/simplification-dataset"  # change this
HF_ACCEPTED_JSON = "core_simp_dataset.json"  # relative path inside the dataset repo
HF_RUN_LOG_JSON = "core_simp_phase1_run_log.json"  # matching experiment's run log
HF_DATASET_REVISION = None  # set a commit hash to freeze the source version
WORK_DIR = Path("/kaggle/working/simplification-qwen3-4b-qlora")
SEED = 42
MAX_LENGTH = 4096
TRAIN = True
RESUME_CHECKPOINT = None
RUN_EVALUATION = True
MAX_EVAL_QUESTIONS = 30  # None evaluates the entire held-out test split
SIMPLIFIER_MAX_NEW_TOKENS = 512
SOLVER_MAX_NEW_TOKENS = 1024
EVAL_TARGET_BENCHMARK = "numina_hard"  # set to the construction benchmark
HF_UPLOAD_ENABLED = False
HF_MODEL_REPO_ID = None  # None uses the helper's default repository name
HF_MODEL_REPO_PRIVATE = True

def optional_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

qlora_config = QLoRAConfig(output_dir=str(WORK_DIR), max_length=MAX_LENGTH, seed=SEED)
WORK_DIR.mkdir(parents=True, exist_ok=True)


## Download the saved construction artifacts

The notebook does not call the construction pipeline. It verifies both downloaded files exist and records their hashes in the data manifest.

In [ ]:
if HF_DATASET_REPO_ID == "owner/simplification-dataset":
    raise ValueError("Set HF_DATASET_REPO_ID to your actual Hugging Face dataset repository.")
hf_read_token = optional_secret("HF_TOKEN")
download_kwargs = {
    "repo_id": HF_DATASET_REPO_ID, "repo_type": "dataset",
    "revision": HF_DATASET_REVISION, "token": hf_read_token,
}
accepted_path = Path(hf_hub_download(filename=HF_ACCEPTED_JSON, **download_kwargs))
log_path = Path(hf_hub_download(filename=HF_RUN_LOG_JSON, **download_kwargs))
del hf_read_token
print({"accepted_json": str(accepted_path), "run_log_json": str(log_path)})

## Audit labels and split by original question

Inspect the class counts and exclusions before starting GPU training. The saved manifest includes the exact source hashes and split membership.

In [ ]:
inspection_tokenizer = AutoTokenizer.from_pretrained(qlora_config.model_name, use_fast=True)
if inspection_tokenizer.pad_token_id is None:
    inspection_tokenizer.pad_token = inspection_tokenizer.eos_token
prepared = prepare_simplification_data(
    accepted_path, log_path, WORK_DIR,
    tokenizer=inspection_tokenizer, max_length=MAX_LENGTH, seed=SEED,
)
training_source = {
    "sources": prepared["manifest"]["sources"],
    "model_name": qlora_config.model_name,
    "seed": SEED, "max_length": MAX_LENGTH,
}
source_path = WORK_DIR / "training_source.json"
if source_path.exists() and json.loads(source_path.read_text(encoding="utf-8")) != training_source:
    raise ValueError("Training inputs changed; use a fresh WORK_DIR instead of reusing an adapter/checkpoint.")
if not save_json_atomic(training_source, str(source_path)):
    raise OSError("Could not save training source identity.")
print(json.dumps({
    "source_rows": prepared["manifest"]["sources"],
    "statuses": prepared["manifest"]["status_counts"],
    "split_label_counts": prepared["manifest"]["split_label_counts"],
    "exclusions": prepared["manifest"]["audit_counts"],
    "token_statistics": prepared["manifest"]["token_statistics"],
}, indent=2))
if not any(r["label_kind"] == "simplify" for r in prepared["splits"]["train"]):
    raise ValueError("Training split has no accepted simplification examples.")
if not any(r["label_kind"] == "copy" for r in prepared["splits"]["train"]):
    raise ValueError("Training split has no copy examples.")

## Load QLoRA and verify one backward step

In [ ]:
model, tokenizer = load_qlora_model(qlora_config)
tokenized, token_report = tokenize_splits(prepared["splits"], tokenizer, MAX_LENGTH)
smoke_loss = smoke_test_training_step(
    model, CompletionOnlyCollator(tokenizer), tokenized["train"][0]
)
print({"smoke_loss": smoke_loss, "trainable_parameters": model.get_nb_trainable_parameters()})

## Train or resume, then optionally upload the best adapter

A trained adapter is saved under `best_adapter`. Upload remains opt-in.

In [ ]:
if TRAIN:
    trainer = train_qlora(
        model, tokenizer, tokenized, qlora_config,
        resume_from_checkpoint=RESUME_CHECKPOINT,
    )
    print("Best checkpoint:", trainer.state.best_model_checkpoint)
    if HF_UPLOAD_ENABLED:
        hf_write_token = optional_secret("HF_TOKEN")
        if not hf_write_token:
            raise RuntimeError("HF_TOKEN with model-write access is required for upload.")
        try:
            upload_result = upload_adapter_to_hub(
                WORK_DIR / "best_adapter", hf_write_token, HF_MODEL_REPO_ID,
                private=HF_MODEL_REPO_PRIVATE,
                commit_message="Upload simplification QLoRA adapter",
            )
        finally:
            del hf_write_token
        print("Uploaded adapter:", upload_result["url"])
elif not (WORK_DIR / "best_adapter" / "adapter_config.json").is_file():
    raise FileNotFoundError("TRAIN=False requires an existing best_adapter.")

## Reload the adapter and generate from base or adapted Qwen

The local solver always runs with the adapter disabled. The evaluator is the repository's existing API grader.

In [ ]:
del model
torch.cuda.empty_cache()
model, tokenizer = load_adapter_for_inference(
    str(WORK_DIR / "best_adapter"), qlora_config.model_name, gpu_index=0,
)
generator = LocalFusionGenerator(model, tokenizer, max_input_tokens=MAX_LENGTH)
settings = {
    "hf_dataset_repo_id": HF_DATASET_REPO_ID,
    "hf_accepted_json": HF_ACCEPTED_JSON, "hf_run_log_json": HF_RUN_LOG_JSON,
    "hf_dataset_revision": HF_DATASET_REVISION, "seed": SEED,
    "max_length": MAX_LENGTH, "max_eval_questions": MAX_EVAL_QUESTIONS,
    "simplifier_max_new_tokens": SIMPLIFIER_MAX_NEW_TOKENS,
    "solver_max_new_tokens": SOLVER_MAX_NEW_TOKENS,
    "eval_target_benchmark": EVAL_TARGET_BENCHMARK,
    "model_name": qlora_config.model_name,
    "source_files": prepared["manifest"]["sources"],
    "evaluator_model": CONFIG["AVALAI_MODEL_NAME_EVALUATOR"],
}
settings_path = WORK_DIR / "evaluation_settings.json"
if settings_path.exists():
    previous_settings = json.loads(settings_path.read_text(encoding="utf-8"))
    comparable = lambda value: {k: v for k, v in value.items() if k != "max_eval_questions"}
    if comparable(previous_settings) != comparable(settings):
        raise ValueError("Evaluation inputs or settings changed; use a fresh WORK_DIR to avoid stale cached results.")
if not save_json_atomic(settings, str(settings_path)):
    raise OSError("Could not save evaluation settings.")

## Held-out simplifier behavior and solver benefit

The first 30 test questions are evaluated by default. Set `MAX_EVAL_QUESTIONS=None` for the full held-out set. Question-level JSON is saved after every case, so rerunning this cell resumes completed records. Only rows with ground truth enter solver-accuracy results. A changed proxy is solved by base Qwen and given as context to base Qwen solving the original question.

In [ ]:
if RUN_EVALUATION:
    from src.api_manager import AvalAIAPIManager

    setup_kaggle_mode("/kaggle/working")
    eval_config = dict(CONFIG)
    eval_config.update({
        "TARGET_BENCHMARK": EVAL_TARGET_BENCHMARK,
        "TARGET_BENCHMARKS": [],
        "EVAL_PARSE_BOXED_GROUND_TRUTH": True,
        "DEFAULT_EVALUATOR_TEMPERATURE": 0.0,
    })
    avalai_key = optional_secret("AVALAI_API_KEY")
    if not avalai_key:
        raise RuntimeError("Set AVALAI_API_KEY in Kaggle Secrets to grade solver outputs.")
    evaluator = AvalAIAPIManager(
        api_key_or_list=[avalai_key], base_url=eval_config["AVALAI_BASE_URL"],
        model_quotas=eval_config["AVALAI_MODEL_QUOTAS"], config=eval_config,
    )
    del avalai_key
    test_rows = prepared["splits"]["test"]
    if MAX_EVAL_QUESTIONS is not None:
        test_rows = test_rows[:MAX_EVAL_QUESTIONS]
    results_path = WORK_DIR / "heldout_evaluation.json"
    previous = json.loads(results_path.read_text(encoding="utf-8")) if results_path.exists() else []
    completed = {row["record_id"]: row for row in previous if isinstance(row, dict) and "record_id" in row}
    selected_ids = {row["record_id"] for row in test_rows}
    completed = {key: value for key, value in completed.items() if key in selected_ids}
    for row in test_rows:
        if row["record_id"] in completed:
            continue
        case = evaluate_question(
            row, generator, evaluator, eval_config, seed=SEED,
            simplifier_max_new_tokens=SIMPLIFIER_MAX_NEW_TOKENS,
            solver_max_new_tokens=SOLVER_MAX_NEW_TOKENS,
        )
        completed[row["record_id"]] = case
        ordered = [completed[item["record_id"]] for item in test_rows if item["record_id"] in completed]
        if not save_json_atomic(ordered, str(results_path)):
            raise OSError("Could not checkpoint held-out evaluation.")
    evaluated = [completed[row["record_id"]] for row in test_rows]
    summary = summarize_evaluation(evaluated)
    if not save_json_atomic(summary, str(WORK_DIR / "heldout_summary.json")):
        raise OSError("Could not save evaluation summary.")
    print(json.dumps(summary, indent=2))
else:
    print("Evaluation skipped; best_adapter and data manifest remain available.")